# **The Automated ML Pipeline Simulation**

**Step 1: Install Dependencies**
Setup & Port Cleaning
Run this to ensure any old servers are killed before starting.

In [1]:
!pip install fastapi uvicorn nest-asyncio
import os

# This command kills any process running on port 8000
!fuser -k 8000/tcp

You should consider upgrading via the 'C:\Users\Mani\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.
'fuser' is not recognized as an internal or external command,
operable program or batch file.


## **Step 2: The Unified Pipeline (Train -> Export -> Deploy)**

Run this block to execute the entire lifecycle. It will train the model on synthetic house data, extract the parameters, and start the API automatically.

The Full Pipeline have modified the server logic to use a Unique Port or check for availability.

In [2]:
import numpy as np
import threading
import time
import uvicorn
import nest_asyncio
import requests
from fastapi import FastAPI
from pydantic import BaseModel

# --- GLOBAL MODEL REPOSITORY (The "Bridge") ---
# In a real system, this would be a file like 'model.pkl' or a Database
production_model = {
    "W": None,
    "B": None,
    "MU": None,
    "SIGMA": None,
    "ready": False
}

# ==========================================
# PHASE 1: MODEL DEVELOPMENT (TRAINING)
# ==========================================
def train_model():
    print("🧠 PHASE 1: Starting Model Training...")
    np.random.seed(42)
    n = 500

    # Generate Synthetic Data
    sqft = np.random.randint(800, 5000, (n, 1))
    age = np.random.randint(0, 50, (n, 1))
    dist = np.random.uniform(1, 20, (n, 1))
    X_raw = np.hstack([sqft, age, dist])
    y = (0.12 * sqft) - (3.0 * age) - (8.0 * dist) + 200 + np.random.randn(n, 1) * 10

    # Feature Scaling
    mu = np.mean(X_raw, axis=0)
    sigma = np.std(X_raw, axis=0)
    X = (X_raw - mu) / sigma

    # Gradient Descent
    w = np.zeros((3, 1))
    b = 0.0
    lr = 0.1
    for _ in range(1000):
        err = (X @ w + b) - y
        w -= lr * (1/n) * (X.T @ err)
        b -= lr * np.mean(err)

    # "Export" parameters to the Production Bridge
    production_model["W"] = w
    production_model["B"] = b
    production_model["MU"] = mu
    production_model["SIGMA"] = sigma
    production_model["ready"] = True
    print("✅ Training Complete. Model parameters exported to Production.\n")

# ==========================================
# PHASE 2: PRODUCTION API (DEPLOYMENT)
# ==========================================
app = FastAPI()

class HouseInput(BaseModel):
    sqft: float
    age: float
    distance: float

@app.post("/predict")
async def predict(item: HouseInput):
    if not production_model["ready"]:
        return {"error": "Model not trained yet"}

    # Retrieve parameters dynamically from the bridge
    w, b = production_model["W"], production_model["B"]
    mu, sigma = production_model["MU"], production_model["SIGMA"]

    # Pre-process & Inference
    x_input = np.array([[item.sqft, item.age, item.distance]])
    x_scaled = (x_input - mu) / sigma
    price = (x_scaled @ w + b)[0][0]

    return {"price_k": round(float(price), 2), "status": "Live Inference"}

# ==========================================
# PHASE 3: EXECUTION (THE DIGITAL TWIN)
# ==========================================
# 1. Run Training
train_model()

# 2. Start API Server in background thread
def run_api():
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2) # Wait for server to boot
print("🚀 PRODUCTION API IS LIVE\n")

# 3. Simulated Client Request
test_house = {"sqft": 3000, "age": 10, "distance": 5}
response = requests.post("http://127.0.0.1:8000/predict", json=test_house)
print(f"📡 Client Sent: {test_house}")
print(f"💰 Server Returned: {response.json()}")

ModuleNotFoundError: No module named 'uvicorn'

**What This Simulates:**

1. The CI/CD Pipeline (Continuous Integration/Deployment)
In a modern tech company, when code is changed, a "Runner" automatically: Trains the model, Tests the accuracy, and Pushes the weights to the API server. Our code simulates this with the model_store dictionary acting as the "Registry."

2. Handling State Mismatch
Notice that the API doesn't have any data of its own. It reaches out to the model_store only when a request arrives. This is how Stateless APIs work—they are ready to change their behavior the moment a new model version is uploaded.

3. The "Cold Start" Problem
If you tried to call the API before train_and_export() finished, it would return an error. This is a real-world concept called a Cold Start, where a service is online but the model hasn't finished loading into memory yet.

Beginner Debugging Tip:

If you see the Errno 98 again:

Go to the top menu: Runtime > Restart Session.

This completely clears the computer's memory and forces all ports to close.

Then run your cells from the top.

**Detailed Explanation for Beginners**

1. The "Bridge" Concept (Decoupling)

In the previous version, we manually typed W = 0.15. Here, the API doesn't know the weights until the train_model() function finishes.

This is Decoupling: the Team that builds the model (Data Scientists) and the Team that builds the API (Software Engineers) work on different parts, connected by a shared "Model Repository."

2. Automatic Parameter Extraction

Notice that the API code now uses variables:w, b = production_model ["W"], production_model ["B"]. This allows the model to be updated. If you re-train the model with more data next month, the API will automatically start using the new, better weights without you changing a single line of the API code.

3. Consistency of Pre-processing

A major failure point in production is using different scaling for training and testing. By storing MU and SIGMA in the production_model dictionary, we ensure the API treats the incoming data exactly the same way the training algorithm treated the training data.

4. The Lifecycle Flow

Data Generation: Simulating real-world house sales.

Training: Learning the patterns ($w$ and $b$).

Serialization (The Export): Saving those patterns into our "bridge."

Inference: Using those patterns to answer a user's request.


Lab Challenge

The students to change the "True Weights" in the training section (e.g., change 0.12 to 0.20).They will see that the API output changes automatically. This proves that the system is truly dynamic and not just "hardcoded."